# Hyperparameter Tuning a CIFAR-10 CNN with the TensorBoard HParams Dashboard

Based on the course TensorBoard Lab 4 (HParams, Fashion-MNIST MLP). Changes made in this version:

| | Original lab | This lab |
|---|---|---|
| Dataset | Fashion-MNIST (grayscale 28x28) | **CIFAR-10** (color 32x32x3) |
| Model | Flatten + 1 Dense layer MLP | **2-block CNN** (Conv2D + MaxPool) with augmentation |
| Hyperparameters | units, dropout, optimizer (8 runs) | **conv filters, dropout, optimizer, learning rate** (16 runs) |
| Metrics | test accuracy only, single step | **per-epoch train/val loss and accuracy** + test loss and accuracy |
| Dashboards | HParams | HParams, **Scalars, Graphs, Images (confusion matrix of best run)** |
| Misc | `!rm -rf` (Unix only), debug dump enabled (slow) | cross-platform log cleanup, no debug dump |


In [1]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard

In [2]:
import io, itertools, shutil
from datetime import datetime

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorboard.plugins.hparams import api as hp

print("TensorFlow version:", tf.__version__)
assert int(tf.__version__.split(".")[0]) >= 2, "This notebook requires TensorFlow 2.x"

tf.random.set_seed(42)
np.random.seed(42)

LOG_DIR = "logs/hparam_tuning"
shutil.rmtree("logs", ignore_errors=True)  # clear logs from previous runs (works on Windows too)

TensorFlow version: 2.21.0


## Data: CIFAR-10
10 classes of 32x32 color images. To keep the 16-run grid search fast on a CPU, we train on a 20k-image subset and keep 5k images for validation. All 10k test images are used for the final score.

In [3]:
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = y_train.flatten(), y_test.flatten()

x_val, y_val = x_train[45000:], y_train[45000:]
x_train, y_train = x_train[:20000], y_train[:20000]
print(x_train.shape, x_val.shape, x_test.shape)

        0/170498071 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

    40960/170498071 ━━━━━━━━━━━━━━━━━━━━ 5:57 2us/step

   114688/170498071 ━━━━━━━━━━━━━━━━━━━━ 3:22 1us/step

   245760/170498071 ━━━━━━━━━━━━━━━━━━━━ 2:09 1us/step

   434176/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:33 1us/step

...

169607168/170498071 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

170082304/170498071 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 22s 0us/step


C:\Users\scrad\AppData\Roaming\Python\Python313\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


(20000, 32, 32, 3) (5000, 32, 32, 3) (10000, 32, 32, 3)


## Hyperparameter search space
Four hyperparameters instead of three; learning rate is new, and conv filters replaces dense units.

In [4]:
HP_FILTERS = hp.HParam("filters", hp.Discrete([16, 32]))
HP_DROPOUT = hp.HParam("dropout", hp.Discrete([0.2, 0.4]))
HP_OPTIMIZER = hp.HParam("optimizer", hp.Discrete(["adam", "sgd"]))
HP_LR = hp.HParam("learning_rate", hp.Discrete([1e-3, 1e-2]))
HPARAMS = [HP_FILTERS, HP_DROPOUT, HP_OPTIMIZER, HP_LR]

METRICS = [
    hp.Metric("test_accuracy", display_name="Test accuracy"),
    hp.Metric("test_loss", display_name="Test loss"),
    hp.Metric("epoch_accuracy", group="validation", display_name="Val accuracy (last epoch)"),
    hp.Metric("epoch_loss", group="validation", display_name="Val loss (last epoch)"),
]

with tf.summary.create_file_writer(LOG_DIR).as_default():
    hp.hparams_config(hparams=HPARAMS, metrics=METRICS)

## Model: small CNN with augmentation

In [5]:
EPOCHS = 5

def build_model(hparams):
    f = hparams[HP_FILTERS]
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(32, 32, 3)),
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.Conv2D(f, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(f * 2, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    opt_cls = {"adam": tf.keras.optimizers.Adam, "sgd": tf.keras.optimizers.SGD}[hparams[HP_OPTIMIZER]]
    model.compile(optimizer=opt_cls(learning_rate=hparams[HP_LR]),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

## One trial
Each run writes to its own directory:
- `tf.keras.callbacks.TensorBoard` logs per-epoch train/validation curves (Scalars) and the conceptual Keras model graph (Graphs).
- `hp.KerasCallback` records the hyperparameter values for the HParams dashboard.
- Test loss and accuracy are logged after training.

In [6]:
def run(run_dir, hparams):
    model = build_model(hparams)
    model.fit(x_train, y_train, epochs=EPOCHS, batch_size=128,
              validation_data=(x_val, y_val), verbose=0,
              callbacks=[tf.keras.callbacks.TensorBoard(run_dir),
                         hp.KerasCallback(run_dir, hparams)])
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    with tf.summary.create_file_writer(run_dir).as_default():
        tf.summary.scalar("test_accuracy", test_acc, step=EPOCHS)
        tf.summary.scalar("test_loss", test_loss, step=EPOCHS)
    return model, test_acc

## Grid search (2 x 2 x 2 x 2 = 16 runs)

In [7]:
results = []
grid = itertools.product(*(h.domain.values for h in HPARAMS))
for i, values in enumerate(grid):
    hparams = dict(zip(HPARAMS, values))
    run_name = f"run-{i:02d}"
    model, acc = run(f"{LOG_DIR}/{run_name}", hparams)
    print(f"{run_name} {({h.name: v for h, v in hparams.items()})} test_acc={acc:.4f}")
    results.append((acc, run_name, hparams, model))

best_acc, best_run, best_hparams, best_model = max(results, key=lambda r: r[0])
print(f"\nBest: {best_run} acc={best_acc:.4f}", {h.name: v for h, v in best_hparams.items()})

run-00 {'filters': 16, 'dropout': 0.2, 'optimizer': 'adam', 'learning_rate': 0.001} test_acc=0.5493


run-01 {'filters': 16, 'dropout': 0.2, 'optimizer': 'adam', 'learning_rate': 0.01} test_acc=0.5252


run-02 {'filters': 16, 'dropout': 0.2, 'optimizer': 'sgd', 'learning_rate': 0.001} test_acc=0.1700


run-03 {'filters': 16, 'dropout': 0.2, 'optimizer': 'sgd', 'learning_rate': 0.01} test_acc=0.3338


run-04 {'filters': 16, 'dropout': 0.4, 'optimizer': 'adam', 'learning_rate': 0.001} test_acc=0.5741


...

run-11 {'filters': 32, 'dropout': 0.2, 'optimizer': 'sgd', 'learning_rate': 0.01} test_acc=0.3418


run-12 {'filters': 32, 'dropout': 0.4, 'optimizer': 'adam', 'learning_rate': 0.001} test_acc=0.5968


run-13 {'filters': 32, 'dropout': 0.4, 'optimizer': 'adam', 'learning_rate': 0.01} test_acc=0.4632


run-14 {'filters': 32, 'dropout': 0.4, 'optimizer': 'sgd', 'learning_rate': 0.001} test_acc=0.1516


run-15 {'filters': 32, 'dropout': 0.4, 'optimizer': 'sgd', 'learning_rate': 0.01} test_acc=0.3345

Best: run-08 acc=0.6299 {'filters': 32, 'dropout': 0.2, 'optimizer': 'adam', 'learning_rate': 0.001}


## Confusion matrix of the best run (Images dashboard)

In [8]:
def confusion_matrix_image(model):
    preds = model.predict(x_test, verbose=0).argmax(axis=1)
    cm = tf.math.confusion_matrix(y_test, preds, num_classes=10).numpy()
    cm = cm / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(10), CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(10), CLASS_NAMES)
    for i, j in itertools.product(range(10), range(10)):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Normalized confusion matrix (best run)")
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)
    return tf.expand_dims(tf.image.decode_png(buf.getvalue(), channels=4), 0)

with tf.summary.create_file_writer(f"{LOG_DIR}/{best_run}").as_default():
    tf.summary.image("confusion_matrix", confusion_matrix_image(best_model), step=EPOCHS)

## Launch TensorBoard
- **HParams** tab: Table View, Parallel Coordinates View, and Scatter Plot Matrix across the 16 runs. Sort by Test accuracy.
- **Scalars / Time Series** tab: per-epoch `epoch_loss` / `epoch_accuracy` for train and validation of every run.
- **Graphs** tab: the conceptual Keras graph of the CNN (select the `keras` tag).
- **Images** tab: confusion matrix for the best run.

Outside Jupyter: `tensorboard --logdir logs/hparam_tuning`, then open http://localhost:6006

In [9]:
%tensorboard --logdir logs/hparam_tuning